In [1]:
import os
import torch

os.environ["CUDA_VISIBLE_DEVICES"] = "4"

from tts.config.ndaligner.training_module_config import NDAlignerTrainingModuleConfigs
from tts.config.utils.io import load_config
from tts.models.ndaligner import init_nd_aligner_training_module
from tts.tokenizer.espeak_tokenizer import ESPEAKTokenizer

/home/blue2959/monotonic_tts/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## INIT Models

In [ ]:

device = 'cuda' if torch.cuda.is_available() else 'cpu'
aligner_training_module_cfg_path = "/home/blue2959/monotonic_tts/runs/nd_aligner_main_vctk+full+win_len=512(orig=1024)_20260807-045023/model_config.json"
aligner_training_module_ckpt_path = "/home/blue2959/monotonic_tts/runs/nd_aligner_main_vctk+full+win_len=512(orig=1024)_20260807-045023/checkpoints_timit_bae/best_step_timit_bae_0.019855_step_150500_epoch_42.pth"

model_config = load_config(aligner_training_module_cfg_path, NDAlignerTrainingModuleConfigs,)

aligner_training_module = init_nd_aligner_training_module(
    config=model_config,
    device=device,
)
aligner_training_module.load_checkpoint(
    ckpt_path=aligner_training_module_ckpt_path,
    device=device,
)
aligner = aligner_training_module.nd_aligner.eval()

Loading nested state_dict from key 'model' in /home/blue2959/monotonic_tts/runs/nd_aligner_main_vctk+full+win_len=512(orig=1024)_20260807-045023/checkpoints_timit_bae/best_step_timit_bae_0.019855_step_150500_epoch_42.pth
✅ All weights matched perfectly.
Checkpoint loading process finished.


In [4]:
TIMIT_ROOT = "/shared/data_zfs/blue2959/TIMIT/TEST"
BUCKEYE_ROOT = "/shared/data_zfs/blue2959/Buckeye-grid" # (compatible with timit benchmarker!)

## INIT BenchMarkers

In [5]:
# from silero_vad import load_silero_vad

# assert aligner.input_maker is not None

# aligner.input_maker.silero_model = load_silero_vad(onnx=True)
# aligner.input_maker.trim_nonspeech_region = aligner.input_maker.zero_nonspeech_region = True

In [6]:
from tts.benchmark.timit.benchmarker import TIMITErrorAnalyzer

assert aligner.input_maker is not None

analyzer = TIMITErrorAnalyzer(
    root_dir=TIMIT_ROOT,
    # root_dir=BUCKEYE_ROOT,
    ref_audio_sr=16_000,
    hyp_audio_sr=model_config.nd_aligner.audio.sr,
    hyp_hop_length=model_config.nd_aligner.audio.hop_length,
    input_maker=aligner.input_maker,
    analysis_dir="./results/timit_error_analysis",
    hyp_ignore_symbols=aligner.input_maker.tokenizer.ignore_symbols,
    max_ref_words_per_hyp_word=5,
    seed=42,
)

[TIMITBenchMarker] Found 1680 valid (WRD, WAV, TXT) triplets.


In [7]:
rows = analyzer.analyze(
    aligner=aligner,
    max_test_samples=None,
    top_k_alignments=50,
    compute_entropy=True,
)

Analyzing TIMIT errors:   0%|          | 0/1680 [00:00<?, ?it/s]

Saving worst alignments: 100%|██████████| 50/50 [00:41<00:00,  1.22it/s]
